In [ ]:
from __future__ import annotations

import csv
import hashlib
import time
from pathlib import Path
import pandas as pd
from dataclasses import dataclass
from datetime import datetime, date, timedelta
from pathlib import Path
from typing import Iterable, Optional, Tuple, List, Dict

import requests
import xml.etree.ElementTree as ET

from zoneinfo import ZoneInfo

import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar


# ============================================================
# CONFIG
# ============================================================

# Prefer env var to avoid hardcoding.
# Example (PowerShell):  setx SMARTERROADS_TOKEN "your_token"
import os
TOKEN = os.environ.get("SMARTERROADS_TOKEN", "").strip()  # Set SMARTERROADS_TOKEN env var before running

BASE = "https://data.511-atis-ttrip-prod.iteriscloud.com/smarterRoads"

YEARS = [2024]
# MONTHS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
MONTHS = [7,8,9,10,11,12]

OUT_ROOT = Path("../../Data/a_raw_download/i66_toll_trip_pricing_monthly")
RAW_DIR = OUT_ROOT / "raw_xml"                  # optional raw XML storage
CSV_DIR = OUT_ROOT / "csv_monthly"              # monthly CSV outputs
PARQUET_DIR = OUT_ROOT / "parquet_monthly"      # monthly parquet outputs
MANIFEST_PATH = OUT_ROOT / "manifest.csv"       # resume log

LOCAL_TZ = ZoneInfo("America/New_York")

# Archive cadence observed in UI: 02 then every 5 minutes
CADENCE_MINUTES = [2, 7, 12, 17, 22, 27, 32, 37, 42, 47, 52, 57]

# Toll windows in local time (end exclusive)
AM_START = (5, 30)   # 05:30
AM_END = (9, 30)     # 09:30
PM_START = (15, 0)   # 15:00
PM_END = (19, 0)     # 19:00

WEEKDAY_ONLY = True
EXCLUDE_US_FED_HOLIDAYS = True

SLEEP_SECONDS_BETWEEN_REQUESTS = 0.25
MAX_RETRIES = 3
TIMEOUT_SECONDS = 45

SAVE_RAW_XML = True

In [2]:

# ============================================================
# MANIFEST
# ============================================================

@dataclass(frozen=True)
class ManifestRow:
    year: int
    month: int
    local_dt: str
    url_dt: str
    url: str
    status: str
    http_code: str
    filepath: str
    bytes: str
    sha256: str
    downloaded_at_local: str
    note: str


def ensure_dirs() -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)
    PARQUET_DIR.mkdir(parents=True, exist_ok=True)


def manifest_exists() -> bool:
    return MANIFEST_PATH.exists()


def append_manifest(row: ManifestRow) -> None:
    new_file = not manifest_exists()
    with MANIFEST_PATH.open("a", newline="", encoding="utf-8") as f:
        fieldnames = [
            "year", "month", "local_dt", "url_dt", "url", "status", "http_code",
            "filepath", "bytes", "sha256", "downloaded_at_local", "note"
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if new_file:
            writer.writeheader()
        writer.writerow({
            "year": row.year,
            "month": row.month,
            "local_dt": row.local_dt,
            "url_dt": row.url_dt,
            "url": row.url,
            "status": row.status,
            "http_code": row.http_code,
            "filepath": row.filepath,
            "bytes": row.bytes,
            "sha256": row.sha256,
            "downloaded_at_local": row.downloaded_at_local,
            "note": row.note,
        })


def load_ok_set_for_month(year: int, month: int) -> set:
    done = set()
    if not MANIFEST_PATH.exists():
        return done

    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            y_raw = (r.get("year") or "").strip()
            m_raw = (r.get("month") or "").strip()

            # Skip blank or malformed rows (common after manual edits)
            if not y_raw or not m_raw:
                continue
            if not y_raw.isdigit() or not m_raw.isdigit():
                continue

            if int(y_raw) != year or int(m_raw) != month:
                continue

            if (r.get("status") or "").strip().lower() == "ok":
                key = (r.get("url_dt") or r.get("utc_dt") or "").strip()
                if key:
                    done.add(key)

    return done


# ============================================================
# DATE / TIME HELPERS
# ============================================================

def month_start_end(year: int, month: int) -> Tuple[date, date]:
    start = date(year, month, 1)
    if month == 12:
        end = date(year + 1, 1, 1) - timedelta(days=1)
    else:
        end = date(year, month + 1, 1) - timedelta(days=1)
    return start, end


def daterange(d0: date, d1: date) -> Iterable[date]:
    d = d0
    while d <= d1:
        yield d
        d += timedelta(days=1)


def us_federal_holidays_for_year(year: int) -> set:
    cal = USFederalHolidayCalendar()
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year}-12-31")
    hol = cal.holidays(start=start, end=end)
    return set([d.date() for d in hol])


def is_in_am_peak(hh: int, mm: int) -> bool:
    cur = hh * 60 + mm
    start = AM_START[0] * 60 + AM_START[1]
    end = AM_END[0] * 60 + AM_END[1]
    return start <= cur < end


def is_in_pm_peak(hh: int, mm: int) -> bool:
    cur = hh * 60 + mm
    start = PM_START[0] * 60 + PM_START[1]
    end = PM_END[0] * 60 + PM_END[1]
    return start <= cur < end


def peak_local_datetimes_for_day(day_local: date) -> List[datetime]:
    """
    Build ONLY the local timestamps that are inside:
      AM: 05:30 <= t < 09:30
      PM: 15:00 <= t < 19:00
    on the observed cadence minutes.
    """
    out: List[datetime] = []

    # AM window
    for hh in range(AM_START[0], AM_END[0] + 1):
        for mm in CADENCE_MINUTES:
            if is_in_am_peak(hh, mm):
                out.append(datetime(day_local.year, day_local.month, day_local.day, hh, mm, tzinfo=LOCAL_TZ))

    # PM window
    for hh in range(PM_START[0], PM_END[0] + 1):
        for mm in CADENCE_MINUTES:
            if is_in_pm_peak(hh, mm):
                out.append(datetime(day_local.year, day_local.month, day_local.day, hh, mm, tzinfo=LOCAL_TZ))

    return out


# ============================================================
# URL + IO
# ============================================================

def build_archive_url_from_local(dt_local: datetime) -> str:
    """
    IMPORTANT.
    The archive folder structure shown in the UI appears to be in local corridor time.
    So we build the archive path using LOCAL time components, not UTC.

    /tollRoad/I66/archive/YYYY/MM/DD/HH/mm/TollingTripPricing_YYYYMMDD_HHMM.xml
    """
    if dt_local.tzinfo is None:
        dt_local = dt_local.replace(tzinfo=LOCAL_TZ)
    else:
        dt_local = dt_local.astimezone(LOCAL_TZ)

    path = (
        f"/tollRoad/I66/archive/"
        f"{dt_local:%Y}/{dt_local:%m}/{dt_local:%d}/{dt_local:%H}/{dt_local:%M}/"
        f"TollingTripPricing_{dt_local:%Y%m%d}_{dt_local:%H%M}.xml"
    )
    return f"{BASE}{path}"


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def raw_out_path(year: int, month: int, dt_local: datetime) -> Path:
    dt_local = dt_local.astimezone(LOCAL_TZ)
    ym_dir = RAW_DIR / f"{year:04d}" / f"{month:02d}"
    ym_dir.mkdir(parents=True, exist_ok=True)
    fname = f"TollingTripPricing_{dt_local:%Y%m%d}_{dt_local:%H%M}.xml"
    return ym_dir / fname


def monthly_fact_paths(year: int, month: int) -> Tuple[Path, Path]:
    csv_path = CSV_DIR / f"{year:04d}" / f"i66_toll_trip_pricing_{year:04d}_{month:02d}.csv"
    pq_path = PARQUET_DIR / f"{year:04d}" / f"i66_toll_trip_pricing_{year:04d}_{month:02d}.parquet"
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    pq_path.parent.mkdir(parents=True, exist_ok=True)
    return csv_path, pq_path


# ============================================================
# NETWORKING
# ============================================================

def fetch_with_retries(session: requests.Session, url: str, params: dict) -> Tuple[Optional[bytes], int, str]:
    last_note = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = session.get(url, params=params, timeout=TIMEOUT_SECONDS, allow_redirects=True)
            code = r.status_code
            if code == 200:
                return r.content, code, "ok"
            if code in (401, 403):
                return None, code, "auth_error"
            if code == 404:
                return None, code, "not_found"
            last_note = f"http_{code}"
        except Exception as e:
            last_note = f"exc:{repr(e)}"

        time.sleep(0.8 * attempt)

    return None, 0, f"failed:{last_note}"


# ============================================================
# XML PARSING
# ============================================================

FACT_COLUMNS = [
    "CalculatedDateTime",
    "IntervalDateTime",
    "IntervalEndDateTime",
    "CorridorID",
    "CorridorName",
    "StartZoneID",
    "StartZoneName",
    "EndZoneID",
    "EndZoneName",
    "ZoneTollRate",
    "source_file",
]


def parse_toll_xml_bytes(xml_bytes: bytes, source_file: str) -> List[Dict]:
    root = ET.fromstring(xml_bytes)
    rows: List[Dict] = []
    for opt in root.findall(".//opt"):
        a = opt.attrib
        rows.append({
            "CalculatedDateTime": a.get("CalculatedDateTime"),
            "IntervalDateTime": a.get("IntervalDateTime"),
            "IntervalEndDateTime": a.get("IntervalEndDateTime"),
            "CorridorID": a.get("CorridorID"),
            "CorridorName": a.get("CorridorName"),
            "StartZoneID": a.get("StartZoneID"),
            "StartZoneName": a.get("StartZoneName"),
            "EndZoneID": a.get("EndZoneID"),
            "EndZoneName": a.get("EndZoneName"),
            "ZoneTollRate": a.get("ZoneTollRate"),
            "source_file": source_file,
        })
    return rows

def build_month_df_from_raw_xml(year: int, month: int) -> pd.DataFrame:
    ym_dir = RAW_DIR / f"{year:04d}" / f"{month:02d}"
    rows = []

    if not ym_dir.exists():
        return pd.DataFrame(columns=FACT_COLUMNS)

    for xml_path in sorted(ym_dir.glob("TollingTripPricing_*.xml")):
        xml_bytes = xml_path.read_bytes()
        rows.extend(parse_toll_xml_bytes(xml_bytes, source_file=xml_path.name))

    df = pd.DataFrame(rows, columns=FACT_COLUMNS)
    if len(df) == 0:
        return df

    return finalize_month_df(df)

def finalize_month_df(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["CalculatedDateTime", "IntervalDateTime", "IntervalEndDateTime"]:
        df[c] = pd.to_datetime(df[c], utc=True, errors="coerce")
    for c in ["CorridorID", "StartZoneID", "EndZoneID"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    df["ZoneTollRate"] = pd.to_numeric(df["ZoneTollRate"], errors="coerce")

    df = df.drop_duplicates(
        subset=["IntervalDateTime", "CorridorID", "StartZoneID", "EndZoneID"],
        keep="last"
    ).sort_values(["IntervalDateTime", "CorridorName", "StartZoneID", "EndZoneID"])
    return df


# ============================================================
# MONTHLY DOWNLOAD + SAVE
# ============================================================

def download_month(year: int, month: int) -> None:
    ensure_dirs()

    start_day, end_day = month_start_end(year, month)
    done = load_ok_set_for_month(year, month)

    holidays = us_federal_holidays_for_year(year) if EXCLUDE_US_FED_HOLIDAYS else set()

    csv_path, pq_path = monthly_fact_paths(year, month)
    # rows_accum: List[Dict] = []

    # # If outputs already exist, load and continue.
    # if pq_path.exists():
    #     df_existing = pd.read_parquet(pq_path)
    #     rows_accum = df_existing.to_dict(orient="records")
    # elif csv_path.exists():
    #     df_existing = pd.read_csv(csv_path)
    #     rows_accum = df_existing.to_dict(orient="records")

    with requests.Session() as session:
        session.headers.update({"User-Agent": "i66-toll-archive-downloader/1.1"})

        for day_local in daterange(start_day, end_day):
            if WEEKDAY_ONLY and day_local.weekday() >= 5:
                continue
            if EXCLUDE_US_FED_HOLIDAYS and (day_local in holidays):
                continue

            # Daily status counters
            day_counts: Dict[str, int] = {}
            day_ok = 0
            day_fail = 0
            day_skip = 0

            local_dts = peak_local_datetimes_for_day(day_local)

            for dt_local in local_dts:
                url_dt_key = dt_local.strftime("%Y-%m-%dT%H:%M:%S%z")
                if url_dt_key in done:
                    continue

                url = build_archive_url_from_local(dt_local)
                content, code, note = fetch_with_retries(session, url, params={"token": TOKEN})

                downloaded_at_local = datetime.now(LOCAL_TZ).strftime("%Y-%m-%dT%H:%M:%S%z")
                local_key = dt_local.strftime("%Y-%m-%dT%H:%M:%S%z")

                code_key = str(code) if code else "0"
                day_counts[code_key] = day_counts.get(code_key, 0) + 1

                if content is not None and code == 200:
                    source_file = f"TollingTripPricing_{dt_local:%Y%m%d}_{dt_local:%H%M}.xml"

                    if SAVE_RAW_XML:
                        fpath = raw_out_path(year, month, dt_local)
                        fpath.write_bytes(content)
                        filepath = str(fpath)
                    else:
                        filepath = ""

                    # rows_accum.extend(parse_toll_xml_bytes(content, source_file=source_file))

                    row = ManifestRow(
                        year=year,
                        month=month,
                        local_dt=local_key,
                        url_dt=url_dt_key,
                        url=url,
                        status="ok",
                        http_code=str(code),
                        filepath=filepath,
                        bytes=str(len(content)),
                        sha256=sha256_bytes(content),
                        downloaded_at_local=downloaded_at_local,
                        note="",
                    )
                    done.add(url_dt_key)
                    day_ok += 1
                else:
                    status = "skip" if note == "not_found" else "fail"
                    if status == "skip":
                        day_skip += 1
                    else:
                        day_fail += 1

                    row = ManifestRow(
                        year=year,
                        month=month,
                        local_dt=local_key,
                        url_dt=url_dt_key,
                        url=url,
                        status=status,
                        http_code=str(code),
                        filepath="",
                        bytes="",
                        sha256="",
                        downloaded_at_local=downloaded_at_local,
                        note=note,
                    )

                append_manifest(row)
                time.sleep(SLEEP_SECONDS_BETWEEN_REQUESTS)

            # Print one line per day
            codes_sorted = ", ".join([f"{k}:{day_counts[k]}" for k in sorted(day_counts.keys(), key=lambda x: int(x) if x.isdigit() else 999999)])
            print(f"{day_local.isoformat()}  ok:{day_ok}  fail:{day_fail}  skip:{day_skip}  codes[{codes_sorted}]")

    # Save monthly outputs
    # df = pd.DataFrame(rows_accum, columns=FACT_COLUMNS)\
    
    # if len(df) == 0:
    #     df.to_csv(csv_path, index=False)
    #     df.to_parquet(pq_path, index=False)
    #     return

    # df = finalize_month_df(df)

    # df.to_parquet(pq_path, index=False)

    # df_csv = df.copy()
    # for c in ["CalculatedDateTime", "IntervalDateTime", "IntervalEndDateTime"]:
    #     df_csv[c] = df_csv[c].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    # df_csv.to_csv(csv_path, index=False)

    df = build_month_df_from_raw_xml(year, month)

    csv_path, pq_path = monthly_fact_paths(year, month)

    df.to_parquet(pq_path, index=False)

    df_csv = df.copy()
    for c in ["CalculatedDateTime", "IntervalDateTime", "IntervalEndDateTime"]:
        df_csv[c] = pd.to_datetime(df_csv[c], utc=True, errors="coerce") \
                    .dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    df_csv.to_csv(csv_path, index=False)


# ============================================================
# RUN YEAR -> MONTH
# ============================================================

def main() -> None:
    if TOKEN == "PUT_YOUR_TOKEN_HERE" or not TOKEN.strip():
        raise RuntimeError("Set SMARTERROADS_TOKEN env var or set TOKEN in the script before running.")

    ensure_dirs()

    for y in YEARS:
        for m in MONTHS:
            print(f"Starting {y}-{m:02d}")
            download_month(y, m)
            print(f"Completed {y}-{m:02d}")

    print("All done.")


if __name__ == "__main__":
    main()


Starting 2024-04
2024-04-01  ok:39  fail:1  skip:0  codes[200:39, 403:1]
2024-04-02  ok:96  fail:0  skip:0  codes[200:96]
2024-04-03  ok:96  fail:0  skip:0  codes[200:96]
2024-04-04  ok:96  fail:0  skip:0  codes[200:96]
2024-04-05  ok:96  fail:0  skip:0  codes[200:96]
2024-04-08  ok:96  fail:0  skip:0  codes[200:96]
2024-04-09  ok:95  fail:1  skip:0  codes[200:95, 403:1]
2024-04-10  ok:96  fail:0  skip:0  codes[200:96]
2024-04-11  ok:96  fail:0  skip:0  codes[200:96]
2024-04-12  ok:96  fail:0  skip:0  codes[200:96]
2024-04-15  ok:96  fail:0  skip:0  codes[200:96]
2024-04-16  ok:96  fail:0  skip:0  codes[200:96]
2024-04-17  ok:90  fail:6  skip:0  codes[200:90, 403:6]
2024-04-18  ok:93  fail:3  skip:0  codes[200:93, 403:3]
2024-04-19  ok:88  fail:8  skip:0  codes[200:88, 403:8]
2024-04-22  ok:17  fail:79  skip:0  codes[200:17, 403:79]
2024-04-23  ok:19  fail:77  skip:0  codes[200:19, 403:77]
2024-04-24  ok:85  fail:11  skip:0  codes[200:85, 403:11]
2024-04-25  ok:85  fail:11  skip:0  cod

KeyboardInterrupt: 